In [0]:
from pyspark.sql.functions import (col, trim, upper, when, lit, coalesce,
    date_format, unix_timestamp, current_timestamp,
    count, avg, sum, round, xxhash64)
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("silver_load_from", "")
dbutils.widgets.text("silver_load_to", "")
dbutils.widgets.dropdown("optimize", "true", ["true", "false"])

load_type = dbutils.widgets.get("load_type")
load_from = dbutils.widgets.get("silver_load_from").strip()
load_to = dbutils.widgets.get("silver_load_to").strip()
run_optimize = dbutils.widgets.get("optimize") == "true"

SILVER = "workspace.default.silver_nyc311_new"
LOG_TABLE = "workspace.default.pipeline_execution_logs_new"

DIM_DATE = "workspace.default.dim_date"
DIM_AGENCY = "workspace.default.dim_agency"
DIM_COMPLAINT = "workspace.default.dim_complaint"
DIM_LOCATION = "workspace.default.dim_location"
DIM_CHANNEL = "workspace.default.dim_channel"
FACT = "workspace.default.fact_service_request"
AGG = "workspace.default.agg_daily_complaints"

In [0]:
expected_log_columns = {
    "load_type": "STRING",
    "rows_updated": "BIGINT",
    "rows_aggregated": "BIGINT",
}

log_columns = set(spark.table(LOG_TABLE).columns)
missing_columns = [
    f"{c} {t}" for c, t in expected_log_columns.items() if c not in log_columns
]
if missing_columns:
    spark.sql(f"ALTER TABLE {LOG_TABLE} ADD COLUMNS ({', '.join(missing_columns)})")

def write_log(start, status, rows_read=0, inserted=0, updated=0,
              rows_aggregated=0, error=None):
    end = datetime.now()

    log = [(
        "Silver-to-Gold", load_type, "Gold pipeline",
        start, end, status, rows_read, inserted, updated, 0,
        str(error)[:3000] if error else None, rows_aggregated
    )]

    schema = """
        layer STRING, load_type STRING, parameter STRING,
        start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
        rows_read BIGINT, rows_inserted BIGINT, rows_updated BIGINT,
        rows_quarantined BIGINT, error_message STRING,
        rows_aggregated BIGINT
    """

    spark.createDataFrame(log, schema).write.format("delta") \
        .mode("append").saveAsTable(LOG_TABLE)

In [0]:
silver_all = spark.table(SILVER)
silver_df = silver_all

if load_from:
    silver_df = silver_df.filter(
        col("load_timestamp") >= lit(load_from).cast("timestamp"))

if load_to:
    silver_df = silver_df.filter(
        col("load_timestamp") < lit(load_to).cast("timestamp"))

if load_type == "incremental" and not load_from:
    last_run = spark.sql(f"""
        SELECT MAX(end_time) AS last_run
        FROM {LOG_TABLE}
        WHERE layer = 'Silver-to-Gold'
          AND status = 'SUCCESS'
    """).first()["last_run"]

    if last_run is not None:
        silver_df = silver_df.filter(col("load_timestamp") > lit(last_run))

rows_read = silver_df.count()

print("load type:", load_type)
print("rows read from silver:", rows_read)

load type: incremental
rows read from silver: 103053


In [0]:
def clean(name, upper_case=False):
    value = trim(col(name).cast("string"))

    if upper_case:
        value = upper(value)

    return when(value.isNull() | (value == ""), "UNKNOWN").otherwise(value)

silver_prepared = (silver_all
    .filter(col("unique_key").isNotNull())
    .withColumn("_agency", clean("agency", True))
    .withColumn("_agency_name", clean("agency_name"))
    .withColumn("_complaint", clean("complaint_type"))
    .withColumn("_descriptor", clean("descriptor"))
    .withColumn("_borough", clean("borough", True))
    .withColumn("_city", clean("city", True))
    .withColumn("_zip", clean("incident_zip"))
    .withColumn("_board", clean("community_board"))
    .withColumn("_district", clean("council_district"))
    .withColumn("_precinct", clean("police_precinct"))
    .withColumn("_channel", clean("open_data_channel_type", True))
    .withColumn("agency_key", xxhash64("_agency", "_agency_name"))
    .withColumn("complaint_key", xxhash64("_complaint", "_descriptor"))
    .withColumn("location_key", xxhash64(
        "_borough", "_city", "_zip", "_board", "_district", "_precinct"))
    .withColumn("channel_key", xxhash64("_channel"))
)

In [0]:
# COMMAND ----------

# Date dimension
dim_date = spark.sql("""
    SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
           d AS full_date,
           year(d) AS year,
           quarter(d) AS quarter,
           month(d) AS month,
           date_format(d, 'MMMM') AS month_name,
           dayofmonth(d) AS day,
           dayofweek(d) AS day_of_week,
           date_format(d, 'EEEE') AS day_name,
           weekofyear(d) AS week_of_year,
           dayofweek(d) IN (1, 7) AS is_weekend
    FROM (
        SELECT explode(sequence(
            to_date('2010-01-01'),
            to_date('2035-12-31'),
            interval 1 day
        )) AS d
    )
""")

unknown_date = spark.sql("""
    SELECT -1 AS date_key, CAST(NULL AS DATE) AS full_date,
           CAST(NULL AS INT) AS year, CAST(NULL AS INT) AS quarter,
           CAST(NULL AS INT) AS month, 'Unknown' AS month_name,
           CAST(NULL AS INT) AS day, CAST(NULL AS INT) AS day_of_week,
           'Unknown' AS day_name, CAST(NULL AS INT) AS week_of_year,
           false AS is_weekend
""")

dim_date = dim_date.unionByName(unknown_date)

dim_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(DIM_DATE)

# Agency dimension
dim_agency = silver_prepared.select(
    "agency_key",
    col("_agency").alias("agency"),
    col("_agency_name").alias("agency_name")
).dropDuplicates(["agency_key"])

dim_agency = dim_agency.withColumn("load_timestamp", current_timestamp())
dim_agency.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(DIM_AGENCY)

# Complaint dimension
dim_complaint = silver_prepared.select(
    "complaint_key",
    col("_complaint").alias("complaint_type"),
    col("_descriptor").alias("descriptor")
).dropDuplicates(["complaint_key"])

dim_complaint = dim_complaint.withColumn("load_timestamp", current_timestamp())
dim_complaint.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(DIM_COMPLAINT)

# Location dimension
dim_location = silver_prepared.select(
    "location_key",
    col("_borough").alias("borough"),
    col("_city").alias("city"),
    col("_zip").alias("incident_zip"),
    col("_board").alias("community_board"),
    col("_district").alias("council_district"),
    col("_precinct").alias("police_precinct")
).dropDuplicates(["location_key"])

dim_location = dim_location.withColumn("load_timestamp", current_timestamp())
dim_location.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(DIM_LOCATION)

# Channel dimension
dim_channel = silver_prepared.select(
    "channel_key",
    col("_channel").alias("channel")
).dropDuplicates(["channel_key"])

dim_channel = dim_channel.withColumn("load_timestamp", current_timestamp())
dim_channel.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(DIM_CHANNEL)

print("dimensions loaded")

dimensions loaded


In [0]:
from pyspark.sql.functions import sha2, concat_ws

start_time = datetime.now()
inserted = 0
updated = 0
rows_aggregated = 0

try:
    fact_source = silver_df.filter(col("unique_key").isNotNull())

    fact_df = (fact_source
        .withColumn("agency_clean", clean("agency", True))
        .withColumn("agency_name_clean", clean("agency_name"))
        .withColumn("complaint_clean", clean("complaint_type"))
        .withColumn("descriptor_clean", clean("descriptor"))
        .withColumn("borough_clean", clean("borough", True))
        .withColumn("city_clean", clean("city", True))
        .withColumn("zip_clean", clean("incident_zip"))
        .withColumn("board_clean", clean("community_board"))
        .withColumn("district_clean", clean("council_district"))
        .withColumn("precinct_clean", clean("police_precinct"))
        .withColumn("channel_clean", clean("open_data_channel_type", True))
    )

    fact_df = (fact_df
        .withColumn("request_key", col("unique_key").cast("string"))
        .withColumn("created_date_key",
                    coalesce(date_format("created_date", "yyyyMMdd").cast("int"), lit(-1)))
        .withColumn("closed_date_key",
                    coalesce(date_format("closed_date", "yyyyMMdd").cast("int"), lit(-1)))
        .withColumn("due_date_key",
                    coalesce(date_format("due_date", "yyyyMMdd").cast("int"), lit(-1)))
        .withColumn("agency_key", xxhash64("agency_clean", "agency_name_clean"))
        .withColumn("complaint_key", xxhash64("complaint_clean", "descriptor_clean"))
        .withColumn("location_key", xxhash64(
            "borough_clean", "city_clean", "zip_clean",
            "board_clean", "district_clean", "precinct_clean"))
        .withColumn("channel_key", xxhash64("channel_clean"))
        .withColumn("status", coalesce(trim(col("status")), lit("UNKNOWN")))
        .withColumn("created_ts", col("created_date"))
        .withColumn("closed_ts", col("closed_date"))
        .withColumn("request_count", lit(1))
        .withColumn("is_closed", when(col("closed_date").isNotNull(), 1).otherwise(0))
        .withColumn("is_closed_late", when(
            col("closed_date").isNotNull() &
            col("due_date").isNotNull() &
            (col("closed_date") > col("due_date")), 1
        ).otherwise(0))
        .withColumn("resolution_hours", when(
            col("created_date").isNotNull() &
            col("closed_date").isNotNull() &
            (col("closed_date") >= col("created_date")),
            (unix_timestamp("closed_date") - unix_timestamp("created_date")) / 3600.0
        ))
        .withColumn("latitude_rounded", round(col("latitude").cast("double"), 3))
        .withColumn("longitude_rounded", round(col("longitude").cast("double"), 3))
    )

    fact_df = fact_df.select(
        "request_key", "created_date_key", "closed_date_key", "due_date_key",
        "agency_key", "complaint_key", "location_key", "channel_key",
        "status", "created_ts", "closed_ts", "request_count",
        "is_closed", "is_closed_late", "resolution_hours",
        "latitude_rounded", "longitude_rounded"
    ).dropDuplicates(["request_key"])

    hash_columns = [c for c in fact_df.columns if c != "request_key"]

    fact_df = fact_df.withColumn(
        "row_hash",
        sha2(concat_ws("||", *[
            coalesce(col(c).cast("string"), lit("")) for c in hash_columns
        ]), 256)
    )

    fact_df = fact_df.withColumn("load_timestamp", current_timestamp())

    fact_df.limit(0).write.format("delta").mode("ignore").saveAsTable(FACT)

    if load_type == "full":
        fact_df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(FACT)
        inserted = fact_df.count()
    elif fact_df.limit(1).count() > 0:
        fact_table = DeltaTable.forName(spark, FACT)
        fact_table.alias("target").merge(
            fact_df.alias("source"),
            "target.request_key = source.request_key"
        ).whenMatchedUpdateAll(
            condition="source.row_hash <> target.row_hash OR target.row_hash IS NULL"
        ).whenNotMatchedInsertAll().execute()

        metrics = fact_table.history(1).first()["operationMetrics"] or {}
        inserted = int(metrics.get("numTargetRowsInserted", 0))
        updated = int(metrics.get("numTargetRowsUpdated", 0))

    print("fact rows inserted:", inserted)
    print("fact rows updated:", updated)

except Exception as e:
    write_log(start_time, "FAILURE", rows_read, inserted, updated,
              rows_aggregated, e)
    raise

fact rows inserted: 103053
fact rows updated: 0


In [0]:
try:
    fact = spark.table(FACT)

    agg_df = fact.groupBy(
        "created_date_key", "agency_key", "complaint_key"
    ).agg(
        count("*").alias("request_count"),
        sum("is_closed").alias("closed_count"),
        sum("is_closed_late").alias("closed_late_count"),
        avg("resolution_hours").alias("avg_resolution_hours")
    ).withColumn("load_timestamp", current_timestamp())

    rows_aggregated = agg_df.count()

    agg_df.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(AGG)

    print("aggregate rows:", rows_aggregated)

except Exception as e:
    write_log(start_time, "FAILURE", rows_read, inserted, updated,
              rows_aggregated, e)
    raise

aggregate rows: 4741


In [0]:
if run_optimize:
    for table_name in [FACT, AGG, DIM_LOCATION, DIM_COMPLAINT]:
        try:
            spark.sql(f"OPTIMIZE {table_name}")
            print("optimized:", table_name)
        except Exception as e:
            print("optimization skipped:", table_name, str(e)[:100])

write_log(start_time, "SUCCESS", rows_read, inserted, updated, rows_aggregated)

print("Gold completed")
print("rows read:", rows_read)
print("rows inserted:", inserted)
print("rows updated:", updated)
print("rows aggregated:", rows_aggregated)

display(spark.table(LOG_TABLE).orderBy(col("start_time").desc()))

optimized: workspace.default.fact_service_request
optimized: workspace.default.agg_daily_complaints
optimized: workspace.default.dim_location
optimized: workspace.default.dim_complaint
Gold completed
rows read: 103053
rows inserted: 103053
rows updated: 0
rows aggregated: 4741


layer,parameter,start_time,end_time,status,rows_read,rows_inserted,rows_quarantined,error_message,rows_aggregated,load_type,rows_updated
Silver-to-Gold,Gold pipeline,2026-10-23T11:07:14.203Z,2026-10-23T11:08:05.025Z,SUCCESS,103053,103053,0,null,4741,incremental,0
Bronze-to-Silver,nyc311_incremental_2026-10-18_2026-10-20_batch_1,2026-10-23T11:00:06.453Z,2026-10-23T11:00:31.731Z,SUCCESS,11552,0,null,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-20_2026-10-22_batch_1,2026-10-23T10:59:09.449Z,2026-10-23T10:59:19.289Z,SUCCESS,12420,12420,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-19_2026-10-21_batch_1,2026-10-23T10:58:57.869Z,2026-10-23T10:59:07.980Z,SUCCESS,12886,12886,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-18_2026-10-20_batch_1,2026-10-23T10:58:45.945Z,2026-10-23T10:58:56.543Z,SUCCESS,11552,11552,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-16_2026-10-18_batch_1,2026-10-23T10:58:34.288Z,2026-10-23T10:58:44.625Z,SUCCESS,11036,11036,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-15_2026-10-17_batch_1,2026-10-23T10:58:23.044Z,2026-10-23T10:58:32.971Z,SUCCESS,11334,11334,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-14_2026-10-16_batch_1,2026-10-23T10:58:10.678Z,2026-10-23T10:58:21.477Z,SUCCESS,11610,11610,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-13_2026-10-15_batch_1,2026-10-23T10:57:58.113Z,2026-10-23T10:58:09.289Z,SUCCESS,11745,11745,0,null,null,null,null
Raw-to-Bronze,nyc311_incremental_2026-10-12_2026-10-14_batch_1,2026-10-23T10:57:45.599Z,2026-10-23T10:57:56.679Z,SUCCESS,9976,9976,0,null,null,null,null


In [0]:
for table_name in [DIM_DATE, DIM_AGENCY, DIM_COMPLAINT, DIM_LOCATION,
                   DIM_CHANNEL, FACT, AGG]:
    print(table_name, spark.table(table_name).count())

duplicates = spark.table(FACT).groupBy("request_key").count() \
    .filter(col("count") > 1).count()

print("duplicate fact keys:", duplicates)

orphans = spark.sql(f"""
    SELECT count(*) AS orphan_rows
    FROM {FACT} f
    LEFT JOIN {DIM_AGENCY} a ON f.agency_key = a.agency_key
    LEFT JOIN {DIM_COMPLAINT} c ON f.complaint_key = c.complaint_key
    LEFT JOIN {DIM_LOCATION} l ON f.location_key = l.location_key
    LEFT JOIN {DIM_CHANNEL} ch ON f.channel_key = ch.channel_key
    LEFT JOIN {DIM_DATE} d ON f.created_date_key = d.date_key
    WHERE a.agency_key IS NULL
       OR c.complaint_key IS NULL
       OR l.location_key IS NULL
       OR ch.channel_key IS NULL
       OR d.date_key IS NULL
""")

display(orphans)

workspace.default.dim_date 9497
workspace.default.dim_agency 14
workspace.default.dim_complaint 760
workspace.default.dim_location 1843
workspace.default.dim_channel 4
workspace.default.fact_service_request 103053
workspace.default.agg_daily_complaints 4741
duplicate fact keys: 0


orphan_rows
0
